# One TradingEnv step

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

This notebook is a software tutorial using bundled example bars or a small deterministic fixture. It is not a trading recommendation or a research performance claim.

The environment is the behavioral boundary between a policy and simulated execution. This notebook builds a small deterministic market, submits one action, and inspects the resulting state and diagnostics.

In [ ]:
import numpy as np
import pandas as pd

from simulation.environment.env import TradingEnv

n = 120
index = pd.date_range("2025-01-02 14:30", periods=n, freq="min", tz="UTC")
rng = np.random.default_rng(12)
close = 100.0 + np.cumsum(rng.normal(0.0, 0.025, n))
bars = pd.DataFrame(
    {
        "open": close,
        "high": close + 0.2,
        "low": close - 0.2,
        "close": close,
        "volume": 1000,
        "session_id": 0,
    },
    index=index,
)
features = pd.DataFrame(
    {
        "last_swing_high": close + 2.0,
        "last_swing_low": close - 2.0,
        "london_high": close + 3.0,
        "london_low": close - 3.0,
        "asian_high": close + 2.5,
        "asian_low": close - 2.5,
        "volume_spike": 1.0,
        "close": close,
    },
    index=index,
)
env = TradingEnv(
    bars,
    features,
    obs_window=10,
    initial_balance=10_000.0,
    risk_frac_range=(0.01, 0.01),
    rr_ratio_range=(2.0, 2.0),
    min_sl_atr_mult=0.0,
    min_sl_points=0.0,
    block_overnight=False,
)
observation, reset_info = env.reset(seed=12)
print("observation keys:", sorted(observation))
print("reset info:", reset_info)

In [ ]:
# The documented baseline discrete action 1 requests a long entry.
observation, reward, terminated, truncated, info = env.step(1)
print(
    {
        "step": env.step_idx,
        "reward": float(reward),
        "terminated": terminated,
        "truncated": truncated,
    }
)
print("info keys:", sorted(info))
print(
    "position:",
    None
    if env.position is None
    else {
        "direction": env.position.direction,
        "entry": env.position.entry_price,
        "stop": env.position.sl_price,
        "target": env.position.tp_price,
    },
)
print("equity:", env.account.equity)
print("entry diagnostics:", env._entry_diag)
assert "equity" in info
assert env.position is not None
assert np.isclose(env.position.entry_price, 100.65838510502367, atol=1e-6)
assert np.isclose(env.position.sl_price, 98.04598810336206, atol=1e-6)
assert np.isclose(float(info["equity"]), 10_000.0, atol=1e-8)
assert np.isclose(float(reward), 0.0, atol=1e-8)

## Step ordering

```{mermaid}
sequenceDiagram
    participant P as Policy
    participant E as TradingEnv
    participant M as Market and account
    P->>E: action from observation
    E->>M: advance market and protect open position
    E->>E: decode and validate entry/risk
    E->>M: fill, account, and reward
    E-->>P: next observation and info diagnostics
```

The exact transition implementation owns ordering and diagnostics. This trace is a software example, not evidence about policy performance. See [environment API](../../api/domains/trading), [environment step architecture](../../architecture/runtime/environment_step), and [deterministic reproduction](../../reproduce/environment_trace.ipynb).